# BERT Final Model for Fake News Detection

This notebook trains a **DistilBERT** classifier on the news headline dataset using an **80/20 train-validation split**.

Important notes:
- Use **minimal preprocessing** for BERT
- Do **not** use stemming, lemmatization, or stopword removal
- Use the original text column as naturally as possible


In [1]:
# Run this once if needed
# !pip install transformers datasets torch accelerate


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)


## 1. Load the data

In [3]:
data = pd.read_csv('/content/training_data_lowercase.csv', sep='\t', names=['label', 'title'])
data.fillna('', inplace=True)

# Keep text natural for BERT
data['text'] = data['title'].astype(str)

print(data.shape)
data.head()


(34152, 3)


,label,title,text
0,0,donald trump sends out embarrassing new year‚s...,donald trump sends out embarrassing new year‚s...
1,0,drunk bragging trump staffer started russian c...,drunk bragging trump staffer started russian c...
2,0,sheriff david clarke becomes an internet joke ...,sheriff david clarke becomes an internet joke ...
3,0,trump is so obsessed he even has obama‚s name ...,trump is so obsessed he even has obama‚s name ...
4,0,pope francis just called out donald trump duri...,pope francis just called out donald trump duri...


## 2. Train / validation split

In [4]:
train_df, val_df = train_test_split(
    data[['text', 'label']],
    test_size=0.2,
    random_state=42,
    stratify=data['label']
)

print('Train shape:', train_df.shape)
print('Validation shape:', val_df.shape)


Train shape: (27321, 2)
Validation shape: (6831, 2)


## 3. Convert to Hugging Face Dataset

In [5]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

train_dataset, val_dataset


(Dataset({
     features: ['text', 'label'],
     num_rows: 27321
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 6831
 }))

## 4. Tokenization

In [6]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/27321 [00:00<?, ? examples/s]

Map:   0%|          | 0/6831 [00:00<?, ? examples/s]

In [7]:
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

train_dataset[0]


{'label': tensor(0),
 'input_ids': tensor([  101, 10643, 16385,  4108,  3099,  2005, 11193,  2000,  3423,  4697,
          3424,  1011, 12010,  2502,  4140,  2854,   102,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,   

## 5. Load pretrained DistilBERT model

In [8]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. Training arguments

In [9]:
training_args = TrainingArguments(
    output_dir='./bert_results',
    eval_strategy='epoch',
    save_strategy='no',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir='./bert_logs',
    logging_steps=100,
    report_to='none'
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 7. Evaluation metric

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}


## 8. Trainer

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


## 9. Train the model

In [12]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.092742,0.050191,0.984043
2,0.019844,0.057868,0.986825


TrainOutput(global_step=3416, training_loss=0.07838672255569375, metrics={'train_runtime': 292.9775, 'train_samples_per_second': 186.506, 'train_steps_per_second': 11.66, 'total_flos': 1809570899340288.0, 'train_loss': 0.07838672255569375, 'epoch': 2.0})

In [13]:
# Evaluate on training set
train_metrics = trainer.evaluate(train_dataset)

# Evaluate on validation set
val_metrics = trainer.evaluate(val_dataset)

print("Training Accuracy:", train_metrics["eval_accuracy"])
print("Training Loss:", train_metrics["eval_loss"])

print("Validation Accuracy:", val_metrics["eval_accuracy"])
print("Validation Loss:", val_metrics["eval_loss"])

Training Accuracy: 0.996083598696973
Training Loss: 0.016848618164658546
Validation Accuracy: 0.9868247694334651
Validation Loss: 0.05786806344985962


## 10. Evaluate on validation set

In [15]:
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=1)

bert_acc = accuracy_score(val_df['label'], preds)
print('BERT Accuracy:', bert_acc)

print('\nClassification Report:')
print(classification_report(val_df['label'], preds))

print('\nConfusion Matrix:')
print(confusion_matrix(val_df['label'], preds))


BERT Accuracy: 0.9868247694334651

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3515
           1       0.99      0.99      0.99      3316

    accuracy                           0.99      6831
   macro avg       0.99      0.99      0.99      6831
weighted avg       0.99      0.99      0.99      6831


Confusion Matrix:
[[3473   42]
 [  48 3268]]
